# Study for the right exam 📚🔬

Imagine this scenario: it's the day of your final exam, and you're all set to tackle your Political History course, having studied it diligently. However, to your horror, you realize you've mixed up the schedule, and it's actually the Science Exam day! 😱

Certainly, we wouldn't wish this kind of academic mix-up on our LLMs! In this notebook, I'll share my approach to putting together a Science dataset for exam day.

This approach uses the competition data and finds specified number of similar questions to it (Using FAISS similarity search). 

I guess using the provided questions as a starting point handicaps the variety of science questions being pulled in. But the hope is by looking for a larger number of similar samples, we will be able to find a good variety of science topics.


____________________



*Notes*: 

- Expand your validation set beyond competition train data
- Or Build a large science-relevant training set
- Tweak *NUM_Qs* to change the size of the final dataset







If you find this notebook helpful, please consider giving it an upvote! 👍




____________________


Big shoutout to Chris Deotte. Check out his amazing datasets here:

- https://www.kaggle.com/datasets/cdeotte/60k-data-with-context-v2
- https://www.kaggle.com/datasets/cdeotte/40k-data-with-context-v2
- https://www.kaggle.com/code/cdeotte/how-to-use-40k-dataset


# Installations

In [ ]:
! pip install -U sentence-transformers
! pip install transformers
! pip install sentencepiece
! pip install accelerate -U
! pip install faiss-gpu

# Load libraries

In [ ]:
# Utility and Configuration
import os
import pickle

# Data Science and Computation
import numpy as np
import pandas as pd
import torch

# Machine Learning and NLP
import faiss
from sentence_transformers import SentenceTransformer

# Progress Tracking
from tqdm.auto import tqdm

# Set CUDA visible devices (optional)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# Data paths
COMP_LOC = '/kaggle/input/kaggle-llm-science-exam/'

CD60_LOC = '/kaggle/input/60k-data-with-context-v2/'

CD40_LOC = '/kaggle/input/40k-data-with-context-v2/'

# Read data

## Validation

In [ ]:
PATH = CD60_LOC + 'train_with_context2.csv'

valid = pd.read_csv(PATH)

print('Validation data size:', valid.shape )
valid.head()

## 60k dataset

In [ ]:
PATH = CD60_LOC + 'all_12_with_context2.csv'
cd60k = pd.read_csv(PATH)

cd60k = cd60k.drop(columns="source")
cd60k = cd60k.fillna('')

cd60k['source'] = '60k'

print('60k data size:', cd60k.shape )
cd60k.head(2)

## 40k dataset

### MMLU

In [ ]:
# MMLU dataset
PATH = CD40_LOC + 'MMLU_17k_with_context2.csv'
MMLU = pd.read_csv(PATH)


MMLU['E'] = None
keepCols = ['prompt', 'context', 'A', 'B', 'C', 'D', 'E', 'answer']
MMLU = MMLU.loc[MMLU.is_question][keepCols]

MMLU['source'] = 'mmlu'

print('MMLU data size:', MMLU.shape )
MMLU.head(2)

### ScienceQA

In [ ]:
PATH = CD40_LOC + 'ScienceQA_with_context2.csv'
ScienceQA = pd.read_csv(PATH)

keepCols = ['prompt', 'context', 'A', 'B', 'C', 'D', 'E', 'answer']

# ScienceQA_3
ScienceQA_3 = ScienceQA.loc[ScienceQA.image.isna() & (ScienceQA.ct==3)][keepCols]

ScienceQA_3['D'] = None
ScienceQA_3['E'] = None

ScienceQA_3['source'] = 'scienceQA'

# ScienceQA_4
ScienceQA_4 = ScienceQA.loc[ScienceQA.image.isna() & (ScienceQA.ct==4)][keepCols]

ScienceQA_4['E'] = None

ScienceQA_4['source'] = 'scienceQA'

print('ScienceQA_3 data size:', ScienceQA_3.shape )
ScienceQA_3.head(2)

In [ ]:
print('ScienceQA_4 data size:', ScienceQA_4.shape )
ScienceQA_4.head(2)

### OpenBook

In [ ]:
PATH = CD40_LOC + 'OpenBook_with_context2.csv'
OpenBook = pd.read_csv(PATH)
OpenBook = OpenBook.loc[OpenBook.is_question]

# Process OpenBook data
OpenBook.drop(columns = ['id', 'is_question'], inplace = True)
OpenBook['E'] = None
OpenBook = OpenBook[[col for col in OpenBook.columns if col not in ['answer']] + ['answer']]

OpenBook['source'] = 'OpenBook'

print('OpenBook data size:', OpenBook.shape )
OpenBook.head(2)

# 100k dataset (60k + 40k)

In [ ]:
cd100k_df = pd.concat([cd60k, MMLU, ScienceQA_3, ScienceQA_4, OpenBook], ignore_index=True)

cd100k_df.drop_duplicates(inplace=True)
cd100k_df = cd100k_df.reset_index(drop=True)

print('100k data size:', cd100k_df.shape)
cd100k_df.head(2)

# Build FAISS index

In [ ]:
# Sentence Embedding

model = SentenceTransformer('sentence-transformers/paraphrase-albert-small-v2')

sentence_embeddings = model.encode(cd100k_df['prompt'])
dim = sentence_embeddings.shape[1]

sentence_embeddings.shape

In [ ]:
# Build FAISS index
index = faiss.IndexFlatL2(dim)
index.add(sentence_embeddings)

# Remove rows similar to validation samples

## Example

In [ ]:
id = 0

print("-"*25)
print("Question from validation set")
print(valid['prompt'][id])
print("-"*25)

n = 5
query = model.encode([valid['prompt'][id]])

scores, inds = index.search(query, n)

for i, ind in enumerate(inds[0]):
    print(f"Score {scores[0][i]} -  {cd100k_df.iloc[ind]['prompt']}")
    print()

That first one definitely seems like it should be removed from our training set. I will use a threshold score of 100 

## Remove samples from train set

In [ ]:
%%capture
remLst = []

# Score threshold
threshold = 100

# Number of similar questions
n = 20

for prompt in tqdm(valid['prompt'].tolist()):

    query = model.encode([prompt])

    scores, inds = index.search(query, n)

    for i in range(0,len(scores[0])):
        if scores[0][i] < threshold:
            remLst.append(inds[0][i])

In [ ]:
remLst = list(set(remLst))
print(f"Samples to remove: {len(remLst)}")

In [ ]:
cd100k_df = cd100k_df.loc[[i for i in range(0,len(cd100k_df)) if i not in remLst]]
cd100k_df = cd100k_df.reset_index(drop=True)
cd100k_df.drop_duplicates(inplace=True)
cd100k_df.shape

# Rebuild FAISS index

In [ ]:
sentence_embeddings = model.encode(cd100k_df['prompt'])
dim = sentence_embeddings.shape[1]

print(sentence_embeddings.shape)

# Build FAISS index
index = faiss.IndexFlatL2(dim)
index.add(sentence_embeddings)

# Non-Science questions

In [ ]:
id = 20

print("-"*25)
print(valid['prompt'][id])
print("-"*25)

n = 10000
query = model.encode([valid['prompt'][id]])

scores, inds = index.search(query, n)

ind = inds[0][-3:]
print(cd100k_df.iloc[ind]['prompt'].values)
print()

These definitely don't seem relevant to our Science Exam!

# Find Science questions

## Example

In [ ]:
id = 0

print("-"*25)
print(valid['prompt'][id])
print("-"*25)

n = 10
query = model.encode([valid['prompt'][id]])

scores, inds = index.search(query, n)

for ind in inds[0]:
    print(cd100k_df.iloc[ind]['prompt'])
    print()

I'm no scientist but those look alright to me

## Find questions

In [ ]:
%%capture

indLst = []

# Number of similar questions
# Tweak this number to build larger science dataset
NUM_Qs = 6

for prompt in tqdm(valid['prompt'].tolist()):

    query = model.encode([prompt])

    scores, inds = index.search(query, NUM_Qs)

    for i in range(0,len(scores[0])):
        indLst.append(inds[0][i])



In [ ]:
indLst = list(set(indLst))
print(f"Samples: {len(indLst)}")

# Science dataset

In [ ]:
final_df = cd100k_df.loc[indLst]
final_df = final_df.reset_index(drop=True)
final_df.drop_duplicates(inplace=True)

print(final_df.shape)
final_df.head(2)

In [ ]:
final_df['source'].value_counts()

In [ ]:
final_df.head(2)

# Save dataset

In [ ]:
final_df.to_csv("Science_1k.csv", index=False)